In [1]:
import requests
import io
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    precision_recall_curve,
    auc,
    precision_score,
    recall_score,
    f1_score
)


DATASET_API_ENDPOINT = (
    "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv"
)


# =====================================================================
# STUDENT TASK — API FETCHING FUNCTION
# =====================================================================

def fetch_transaction_data(url: str) -> pd.DataFrame:
    """
    Fetch transaction/telemetry data from an HTTP CSV endpoint.

    Reasoning:
    1. requests.get() sends the HTTP GET request.
    2. raise_for_status() prevents HTTP errors from being treated as data.
    3. StringIO makes the response text file-like.
    4. read_csv() converts the CSV payload into a DataFrame.
    5. RequestException provides explicit network/API error handling.
    """

    try:
        response = requests.get(url, timeout=30)

        # Raise an exception for HTTP 4xx/5xx responses.
        response.raise_for_status()

        # Convert CSV response text into a file-like object.
        csv_buffer = io.StringIO(response.text)

        # Parse CSV into Pandas.
        transaction_df = pd.read_csv(csv_buffer)

        return transaction_df

    except requests.exceptions.RequestException as error:
        raise RuntimeError(
            f"Transaction API request failed: {error}"
        ) from error

    except pd.errors.ParserError as error:
        raise RuntimeError(
            f"Transaction CSV parsing failed: {error}"
        ) from error


# =====================================================================
# COMPLETE PIPELINE
# =====================================================================

def run_pipeline():
    """
    Run the complete fraud-detection pipeline.

    Returns a dictionary containing the processed dataset,
    train/test partitions, models, predictions, and metrics.
    """

    # =================================================================
    # TASK 1 — LIVE DATA INGESTION & IMBALANCED LABEL CREATION
    # =================================================================

    raw_df = fetch_transaction_data(
        DATASET_API_ENDPOINT
    )

    if "species" not in raw_df.columns:
        raise ValueError(
            "The API dataset must contain a 'species' column."
        )

    # Create fraud label:
    # setosa = fraud (1)
    # all other species = legitimate (0)
    raw_df["Is_Fraud"] = (
        raw_df["species"] == "setosa"
    ).astype(int)

    # Use the four numeric Iris measurements as model features.
    feature_columns = [
        "sepal_length",
        "sepal_width",
        "petal_length",
        "petal_width"
    ]

    # -----------------------------------------------------------------
    # Create the requested 100 legitimate / 10 fraud dataset.
    #
    # The source contains 100 non-setosa and 50 setosa records.
    # We randomly downsample the fraud class to 10 records.
    # random_state=42 makes the experiment reproducible.
    # -----------------------------------------------------------------

    legitimate_df = raw_df[
        raw_df["Is_Fraud"] == 0
    ].copy()

    fraud_df = raw_df[
        raw_df["Is_Fraud"] == 1
    ].copy()

    fraud_df = fraud_df.sample(
        n=10,
        random_state=42
    )

    imbalanced_df = pd.concat(
        [
            legitimate_df,
            fraud_df
        ],
        ignore_index=True
    )

    # Shuffle after combining classes so class ordering does not
    # accidentally determine train/test observations.
    imbalanced_df = imbalanced_df.sample(
        frac=1,
        random_state=42
    ).reset_index(drop=True)

    X = imbalanced_df[feature_columns]
    y = imbalanced_df["Is_Fraud"]

    # =================================================================
    # TASK 2 — STRATIFIED TRAIN-TEST PARTITIONING
    # =================================================================

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        stratify=y,
        random_state=42
    )

    # =================================================================
    # TASK 3 — STANDARD VS CLASS-WEIGHTED RANDOM FOREST
    # =================================================================

    # Baseline model: all classes receive their normal frequency.
    unweighted_model = RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )

    # Cost-sensitive model: the minority class receives a larger
    # effective penalty during tree construction.
    weighted_model = RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        random_state=42
    )

    # Train both models using exactly the same training data.
    unweighted_model.fit(
        X_train,
        y_train
    )

    weighted_model.fit(
        X_train,
        y_train
    )

    # Predictions for the held-out test set.
    unweighted_predictions = unweighted_model.predict(
        X_test
    )

    weighted_predictions = weighted_model.predict(
        X_test
    )

    # Fraud-class metrics.
    unweighted_precision = precision_score(
        y_test,
        unweighted_predictions,
        pos_label=1,
        zero_division=0
    )

    unweighted_recall = recall_score(
        y_test,
        unweighted_predictions,
        pos_label=1,
        zero_division=0
    )

    unweighted_f1 = f1_score(
        y_test,
        unweighted_predictions,
        pos_label=1,
        zero_division=0
    )

    weighted_precision = precision_score(
        y_test,
        weighted_predictions,
        pos_label=1,
        zero_division=0
    )

    weighted_recall = recall_score(
        y_test,
        weighted_predictions,
        pos_label=1,
        zero_division=0
    )

    weighted_f1 = f1_score(
        y_test,
        weighted_predictions,
        pos_label=1,
        zero_division=0
    )

    # =================================================================
    # PRECISION-RECALL CURVE / AUC
    # =================================================================

    # Random Forest probabilities are required for a Precision-Recall
    # curve; class predictions alone only provide one operating point.
    unweighted_probabilities = unweighted_model.predict_proba(
        X_test
    )[:, 1]

    weighted_probabilities = weighted_model.predict_proba(
        X_test
    )[:, 1]

    unweighted_precision_curve, unweighted_recall_curve, _ = (
        precision_recall_curve(
            y_test,
            unweighted_probabilities
        )
    )

    weighted_precision_curve, weighted_recall_curve, _ = (
        precision_recall_curve(
            y_test,
            weighted_probabilities
        )
    )

    unweighted_pr_auc = auc(
        unweighted_recall_curve,
        unweighted_precision_curve
    )

    weighted_pr_auc = auc(
        weighted_recall_curve,
        weighted_precision_curve
    )

    return {
        "raw_df": raw_df,
        "dataset": imbalanced_df,
        "features": feature_columns,
        "target": "Is_Fraud",

        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,

        "unweighted_model": unweighted_model,
        "weighted_model": weighted_model,

        "unweighted_predictions": unweighted_predictions,
        "weighted_predictions": weighted_predictions,

        "unweighted_precision": unweighted_precision,
        "unweighted_recall": unweighted_recall,
        "unweighted_f1": unweighted_f1,

        "weighted_precision": weighted_precision,
        "weighted_recall": weighted_recall,
        "weighted_f1": weighted_f1,

        "unweighted_pr_auc": unweighted_pr_auc,
        "weighted_pr_auc": weighted_pr_auc
    }


# =====================================================================
# MAIN EXECUTION
# =====================================================================

if __name__ == "__main__":

    results = run_pipeline()

    dataset = results["dataset"]
    X_train = results["X_train"]
    X_test = results["X_test"]
    y_train = results["y_train"]
    y_test = results["y_test"]

    # =================================================================
    # TASK 1 OUTPUT
    # =================================================================

    print(
        "\n========== "
        "TASK 1: LIVE DATA INGESTION & IMBALANCE "
        "==========\n"
    )

    print(
        "[TASK 1 SUCCESS] "
        f"Transaction Data Ingested & Downsampled "
        f"({len(dataset)} Records)"
    )

    print("\nImbalanced Class Distribution:")

    print(
        dataset["Is_Fraud"].value_counts().sort_index()
    )

    print(
        "\nClass Percentage Distribution:"
    )

    print(
        (
            dataset["Is_Fraud"]
            .value_counts(normalize=True)
            .sort_index()
            * 100
        ).round(2)
    )

    # =================================================================
    # TASK 2 OUTPUT
    # =================================================================

    print(
        "\n========== "
        "TASK 2: STRATIFIED TRAIN-TEST PARTITIONING "
        "==========\n"
    )

    print(
        "[TASK 2 SUCCESS] "
        "Data Partitioned with Stratification:"
    )

    print(
        f"Train Set: {len(X_train)} Samples "
        f"({(y_train == 0).sum()} Legitimate / "
        f"{(y_train == 1).sum()} Fraud)"
    )

    print(
        f"Test Set : {len(X_test)} Samples "
        f"({(y_test == 0).sum()} Legitimate / "
        f"{(y_test == 1).sum()} Fraud)"
    )

    # =================================================================
    # TASK 3 OUTPUT
    # =================================================================

    print(
        "\n========== "
        "TASK 3: RANDOM FOREST MODEL COMPARISON "
        "==========\n"
    )

    print(
        "[TASK 3 SUCCESS] Model Comparison Complete:"
    )

    print("\nUnweighted Baseline Model:")

    print(
        f"- Precision (Fraud) : "
        f"{results['unweighted_precision']:.4f}"
    )

    print(
        f"- Recall (Fraud)    : "
        f"{results['unweighted_recall']:.4f}"
    )

    print(
        f"- F1-Score (Fraud)  : "
        f"{results['unweighted_f1']:.4f}"
    )

    print("\nClass-Weighted Model (class_weight='balanced'):")

    print(
        f"- Precision (Fraud) : "
        f"{results['weighted_precision']:.4f}"
    )

    print(
        f"- Recall (Fraud)    : "
        f"{results['weighted_recall']:.4f}"
    )

    print(
        f"- F1-Score (Fraud)  : "
        f"{results['weighted_f1']:.4f}"
    )

    # =================================================================
    # PRECISION-RECALL METRIC OUTPUT
    # =================================================================

    print(
        "\nPrecision-Recall AUC:"
    )

    print(
        f"- Unweighted PR-AUC : "
        f"{results['unweighted_pr_auc']:.4f}"
    )

    print(
        f"- Weighted PR-AUC   : "
        f"{results['weighted_pr_auc']:.4f}"
    )

    # =================================================================
    # FINAL OUTPUT
    # =================================================================

    print(
        "\n========== "
        "IMBALANCED FRAUD DETECTION VIA "
        "CLASS-WEIGHTED RANDOM FORESTS "
        "==========\n"
    )

    print(
        "Data Ingestion Status      : "
        "REST API Ingestion Successful (HTTP 200 OK)"
    )

    fraud_percentage = (
        dataset["Is_Fraud"].mean() * 100
    )

    legitimate_percentage = (
        100 - fraud_percentage
    )

    print(
        f"Master Dataset Records     : "
        f"{len(dataset)} Records "
        f"(Synthetically Imbalanced "
        f"{legitimate_percentage:.1f}% / "
        f"{fraud_percentage:.1f}%)"
    )

    print(
        "Target Output              : "
        "Is_Fraud "
        "(0 = Legitimate, 1 = Fraudulent)"
    )

    print(
        "\nPerformance Metrics Comparison:"
    )

    print(
        "+---------------------+-----------+--------+----------+"
    )

    print(
        "| Model Type          | Precision | Recall | F1-Score |"
    )

    print(
        "+---------------------+-----------+--------+----------+"
    )

    print(
        f"| Standard Unweighted | "
        f"{results['unweighted_precision']:.4f}    | "
        f"{results['unweighted_recall']:.4f}  | "
        f"{results['unweighted_f1']:.4f}   |"
    )

    print(
        f"| Cost-Sensitive      | "
        f"{results['weighted_precision']:.4f}    | "
        f"{results['weighted_recall']:.4f}  | "
        f"{results['weighted_f1']:.4f}   |"
    )

    print(
        "+---------------------+-----------+--------+----------+"
    )

    print("\nConclusion:")

    print(
        "Class-weighted learning changes the training objective so "
        "mistakes involving the minority fraud class receive greater "
        "weight. This allows the Random Forest to treat the rare class "
        "as more important without creating synthetic fraud records."
    )



========== TASK 1: LIVE DATA INGESTION & IMBALANCE ==========

[TASK 1 SUCCESS] Transaction Data Ingested & Downsampled (110 Records)

Imbalanced Class Distribution:
Is_Fraud
0    100
1     10
Name: count, dtype: int64

Class Percentage Distribution:
Is_Fraud
0    90.91
1     9.09
Name: proportion, dtype: float64

========== TASK 2: STRATIFIED TRAIN-TEST PARTITIONING ==========

[TASK 2 SUCCESS] Data Partitioned with Stratification:
Train Set: 88 Samples (80 Legitimate / 8 Fraud)
Test Set : 22 Samples (20 Legitimate / 2 Fraud)

========== TASK 3: RANDOM FOREST MODEL COMPARISON ==========

[TASK 3 SUCCESS] Model Comparison Complete:

Unweighted Baseline Model:
- Precision (Fraud) : 1.0000
- Recall (Fraud)    : 1.0000
- F1-Score (Fraud)  : 1.0000

Class-Weighted Model (class_weight='balanced'):
- Precision (Fraud) : 1.0000
- Recall (Fraud)    : 1.0000
- F1-Score (Fraud)  : 1.0000

Precision-Recall AUC:
- Unweighted PR-AUC : 1.0000
- Weighted PR-AUC   : 1.0000

========== IMBALANCED FRAU